# Cross Validation

### Importing Libraries

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [11]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from src.config import SEED, MODEL_DIR
from src.data import load_train, get_split
from src.models import make_lightgbm
from src.preprocessing import learn_preprocessing, apply_preprocessing
from src.evaluate import cross_validate, report


### Reusing the same LightGBM - Numeric and no scale

In [3]:
X, y = load_train()
X_tr, X_val, y_tr, y_val = get_split(X, y)
print(X_tr.shape, X_val.shape)

(559708, 21) (139927, 21)


In [4]:
state = learn_preprocessing(X_tr, ratings="numeric", scaling="no")
X_tr_encoded = apply_preprocessing(X_tr, state)
X_val_encoded = apply_preprocessing(X_val, state)

In [5]:
scores = cross_validate(X, y, lambda: make_lightgbm(num_leaves=63))

Fold 1: 0.95878
Fold 2: 0.95740
Fold 3: 0.95915
Fold 4: 0.95833
Fold 5: 0.95879
Mean: 0.95849  Std: 0.00061


In [9]:
model = make_lightgbm(num_leaves=63)
model.fit(X_tr_encoded, y_tr)

,num_leaves,63
,learning_rate,0.05
,n_estimators,500
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


- how many times that column was used as a split point across all the trees LightGBM built.

In [7]:
importance = pd.Series(model.feature_importances_, index=X_tr_encoded.columns)
print(importance.sort_values(ascending=False).head(15))

Flight Distance                 2122
Age                             1780
Inflight wifi service           1062
Online boarding                  837
Baggage handling                 780
Checkin service                  692
Seat comfort                     618
On-board service                 578
Gate location                    570
Cleanliness                      520
Class_Business                   519
Leg room service                 512
Ease of Online booking           490
Customer Type_Loyal Customer     483
Inflight entertainment           476
dtype: int32


- how much did splits on this column reduce the model’s error?

In [8]:
gain = model.booster_.feature_importance(importance_type="gain")
importance_gain = pd.Series(gain, index=X_tr_encoded.columns)
print(importance_gain.sort_values(ascending=False).head(15))

Online boarding                   2.866912e+06
Type of Travel_Business travel    6.531981e+05
Inflight wifi service             5.154467e+05
Class_Business                    2.075905e+05
Customer Type_Loyal Customer      1.966001e+05
Inflight entertainment            1.489512e+05
Checkin service                   1.222295e+05
Baggage handling                  1.131249e+05
Inflight wifi service_unrated     1.015973e+05
On-board service                  8.566108e+04
Seat comfort                      7.410870e+04
Cleanliness                       4.541513e+04
Age                               4.431321e+04
Gate location                     3.551032e+04
Ease of Online booking            2.660825e+04
dtype: float64


- Flight Distance was #1 by split count and doesn’t make the gain top 15. Age was #2 by count and drops to #13 by gain. That confirms the earlier suspicion: continuous columns get split often because they offer many cut points, not because they’re valuable.

In [10]:
train_auc = roc_auc_score(
    y_tr, model.predict_proba(X_tr_encoded)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9676


In [12]:
auc = report(model, X_val_encoded, y_val)

Validation ROC-AUC : 0.9576
Validation accuracy: 0.9243

Confusion matrix:
 [[73423  4436]
 [ 6157 55911]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927



In [13]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({"state": state, "model": model}, MODEL_DIR / "lightgbm_63leaves.joblib")

['C:\\Users\\Bharat\\Desktop\\remaining-2026\\Career-Tracks\\Track 01 - Data Science\\Relevant Projects\\airline-satisfaction-prediction\\models\\lightgbm_63leaves.joblib']

##### Trying with 127 leaves

In [14]:
model = make_lightgbm(num_leaves=127)
model.fit(X_tr_encoded, y_tr)

,num_leaves,127
,learning_rate,0.05
,n_estimators,500
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [15]:
train_auc = roc_auc_score(
    y_tr, model.predict_proba(X_tr_encoded)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9738


In [16]:
auc = report(model, X_val_encoded, y_val)

Validation ROC-AUC : 0.9575
Validation accuracy: 0.9250

Confusion matrix:
 [[73465  4394]
 [ 6106 55962]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.93      0.92      0.92    139927
weighted avg       0.93      0.92      0.92    139927



In [17]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({"state": state, "model": model}, MODEL_DIR / "lightgbm_127leaves.joblib")

['C:\\Users\\Bharat\\Desktop\\remaining-2026\\Career-Tracks\\Track 01 - Data Science\\Relevant Projects\\airline-satisfaction-prediction\\models\\lightgbm_127leaves.joblib']

- slight drop in validation auc, signs of overfitting, stick to 63 leaves

In [18]:
scores = cross_validate(
    X, y,
    lambda: make_lightgbm(num_leaves=63, learning_rate=0.025, n_estimators=1000)
)

Fold 1: 0.95881
Fold 2: 0.95734
Fold 3: 0.95925
Fold 4: 0.95853
Fold 5: 0.95881
Mean: 0.95855  Std: 0.00065


In [20]:
model = make_lightgbm(num_leaves=63, learning_rate=0.025, n_estimators=1000)
model.fit(X_tr_encoded, y_tr)


,num_leaves,63
,learning_rate,0.025
,n_estimators,1000
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [21]:
train_auc = roc_auc_score(
    y_tr, model.predict_proba(X_tr_encoded)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9677


In [22]:
auc = report(model, X_val_encoded, y_val)

Validation ROC-AUC : 0.9576
Validation accuracy: 0.9246

Confusion matrix:
 [[73417  4442]
 [ 6115 55953]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927



In [23]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({"state": state, "model": model}, MODEL_DIR / "lightgbm_0.025learn.joblib")

['C:\\Users\\Bharat\\Desktop\\remaining-2026\\Career-Tracks\\Track 01 - Data Science\\Relevant Projects\\airline-satisfaction-prediction\\models\\lightgbm_0.025learn.joblib']